# STA6130Z Week 9 Group Assignment Notebook
## Target Transformations and Machine Learning Forecasting Models
**Course:** STA6130Z Time Series Analysis | USIU-Africa  
**Dataset:** Store Item Demand Forecasting Challenge (Kaggle), Store 1, Item 1  
**Members:** Gabriel Kariuki, Doris Chebet, David Mwiti, Munira Dahir

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

# Settings
STORE, ITEM = 1, 1
H = 1
TRAIN_FRAC = 0.80
SEED = 42

### 1. Load Data and Filter Series

In [2]:
import os, glob, warnings
import numpy as np
import pandas as pd

def find_power_csv():
    # Looks for your file name first, with a fallback
    for path in ['power_usage_2016_to_2020.csv', 'train.csv']:
        if os.path.exists(path):
            return path
    raise FileNotFoundError('power_usage_2016_to_2020.csv not found. Ensure it is in your working directory.')

raw_path = find_power_csv()

# 1. Peek at the first few rows to inspect column names safely
raw_peek = pd.read_csv(raw_path, nrows=5)
print("Columns found in CSV:", raw_peek.columns.tolist())

# 2. Automatically find the date/time column (checking for 'date', 'time', or fallback to the first column)
date_col = next((c for c in raw_peek.columns if 'date' in c.lower() or 'time' in c.lower()), raw_peek.columns[0])
print(f"Automatically identified date column: '{date_col}'")

# 3. Load the dataset using the correct column name
raw = pd.read_csv(raw_path, parse_dates=[date_col])

# 4. Aggregate hourly readings to total daily consumption in kWh
raw['date_only'] = pd.to_datetime(raw[date_col]).dt.date

# Automatically pick the numeric kWh consumption column (ignoring date and metadata columns)
exclude_cols = [date_col, 'date_only', 'weekday', 'weekend', 'vacation', 'COVID_lockdown']
value_col = [c for c in raw.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(raw[c])][0]
print(f"Aggregating daily consumption using value column: '{value_col}'")

s = raw.groupby('date_only')[value_col].sum().astype(float)
s.index = pd.to_datetime(s.index)
s = s.asfreq('D')

print(f'\nSuccessfully Loaded Daily Series: {len(s)} observations ({s.index.min().date()} to {s.index.max().date()})')
print(s.describe().round(2))

Columns found in CSV: ['StartDate', 'Value (kWh)', 'day_of_week', 'notes']
Automatically identified date column: 'StartDate'
Aggregating daily consumption using value column: 'Value (kWh)'

Successfully Loaded Daily Series: 1797 observations (2016-01-06 to 2020-12-06)
count    1498.00
mean       21.40
std        12.64
min         4.61
25%        10.83
50%        16.83
75%        30.73
max        78.23
Name: Value (kWh), dtype: float64


### 2. Explore Target Series & Plot